# Gold Layer — Agregate analitice

**Scop:** Construirea celor 3 tabele agregate pentru dashboards și rapoarte:
1. **`kpi_transactions_daily`** — KPI tranzacționale (zi × branch × canal × tip × valută)
2. **`customer_segments_rfm`** — segmentare clienți pe model RFM (Recency, Frequency, Monetary)
3. **`report_pnl_monthly`** — raport profit & loss lunar per sucursală

**Diferența față de 03a (dimensiuni):**
- 03a = un rând per entitate (client/produs/risc)
- 03b = **agregate** — multe tranzacții reduse la o singură măsurătoare per cheie

**Strategie:** `overwrite` la fiecare rulare (recalculare completă din Silver).

## 1. Configurare

In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, current_date,
    count, sum as f_sum, avg, max as f_max, min as f_min,
    countDistinct, datediff, year, month,
    coalesce, expr, round as f_round, floor,
    least, greatest, ntile, percent_rank,
    date_format, date_trunc, to_date,
)
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

CATALOG = "banking"
SILVER  = "silver"
GOLD    = "gold"

print(f"Silver in:  {CATALOG}.{SILVER}.*")
print(f"Gold out:   {CATALOG}.{GOLD}.*")

Silver in:  banking.silver.*
Gold out:   banking.gold.*


## 2. `kpi_transactions_daily` — KPI tranzacționale detaliate

**Granularitate:** `date_id × branch_id × channel_code × type_code × currency_code`

**Măsurători calculate:**
- `tx_count` — număr total tranzacții
- `tx_count_completed` / `tx_count_failed` — split pe status
- `tx_volume_total_lcy` / `tx_volume_total_ron` — volum total în valuta originală și RON
- `avg_amount_ron` / `min_amount_ron` / `max_amount_ron` — distribuția sumelor
- `unique_customers` / `unique_accounts` — câți clienți/conturi distincte
- `failure_rate` — % tranzacții eșuate (KPI critic pentru operațiuni)

In [0]:
print("Construire kpi_transactions_daily...")

# Pornim de la fact_transactions
fact = spark.table(f"{CATALOG}.{SILVER}.fact_transactions")

# Trebuie sa aducem branch_id prin JOIN cu dim_accounts (fact_transactions are doar account_id)
df = (fact.alias("ft")
    .join(
        spark.table(f"{CATALOG}.{SILVER}.dim_accounts").select("account_id", "branch_id").alias("acc"),
        col("ft.account_id") == col("acc.account_id"),
        "left"
    )
    .select(
        "ft.date_id",
        col("acc.branch_id"),
        "ft.channel_code",
        "ft.type_code",
        "ft.currency_code",
        "ft.transaction_id",
        "ft.account_id",
        "ft.customer_id",
        "ft.amount",
        "ft.amount_ron",
        "ft.status_code",
    )
)

# Agregare pe granularitatea ceruta
kpi_daily = (df
    .groupBy("date_id", "branch_id", "channel_code", "type_code", "currency_code")
    .agg(
        # Count-uri
        count("transaction_id").alias("tx_count"),
        f_sum(when(col("status_code") == "COMPLETED", 1).otherwise(0)).alias("tx_count_completed"),
        f_sum(when(col("status_code") == "FAILED",    1).otherwise(0)).alias("tx_count_failed"),
        f_sum(when(col("status_code") == "REVERSED", 1).otherwise(0)).alias("tx_count_reversed"),
        f_sum(when(col("status_code") == "PENDING",   1).otherwise(0)).alias("tx_count_pending"),
        
        # Volume (doar pe COMPLETED)
        f_sum(when(col("status_code") == "COMPLETED", col("amount")).otherwise(0)).alias("tx_volume_total_lcy"),
        f_sum(when(col("status_code") == "COMPLETED", col("amount_ron")).otherwise(0)).alias("tx_volume_total_ron"),
        
        # Statistici sume (pe completate)
        avg(when(col("status_code") == "COMPLETED", col("amount_ron"))).alias("avg_amount_ron"),
        f_min(when(col("status_code") == "COMPLETED", col("amount_ron"))).alias("min_amount_ron"),
        f_max(when(col("status_code") == "COMPLETED", col("amount_ron"))).alias("max_amount_ron"),
        
        # Diversitate
        countDistinct("customer_id").alias("unique_customers"),
        countDistinct("account_id").alias("unique_accounts"),
    )
)

# Calculăm failure_rate ca metric derivat
kpi_daily = (kpi_daily
    .withColumn("failure_rate",
        when(col("tx_count") > 0,
             f_round(col("tx_count_failed") / col("tx_count") * 100.0, 2))
        .otherwise(lit(0.0)))
    .withColumn("completion_rate",
        when(col("tx_count") > 0,
             f_round(col("tx_count_completed") / col("tx_count") * 100.0, 2))
        .otherwise(lit(0.0)))
    # Round la valori monetare
    .withColumn("tx_volume_total_lcy", f_round(col("tx_volume_total_lcy"), 2))
    .withColumn("tx_volume_total_ron", f_round(col("tx_volume_total_ron"), 2))
    .withColumn("avg_amount_ron",      f_round(col("avg_amount_ron"), 2))
    .withColumn("min_amount_ron",      f_round(col("min_amount_ron"), 2))
    .withColumn("max_amount_ron",      f_round(col("max_amount_ron"), 2))
    .withColumn("gold_loaded_at", current_timestamp())
)

# Reordonăm coloanele
kpi_daily = kpi_daily.select(
    "date_id", "branch_id", "channel_code", "type_code", "currency_code",
    "tx_count", "tx_count_completed", "tx_count_failed", "tx_count_reversed", "tx_count_pending",
    "tx_volume_total_lcy", "tx_volume_total_ron",
    "avg_amount_ron", "min_amount_ron", "max_amount_ron",
    "unique_customers", "unique_accounts",
    "failure_rate", "completion_rate",
    "gold_loaded_at",
)

# Scriem in Gold
(kpi_daily.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.kpi_transactions_daily"))

n_rows = kpi_daily.count()
print(f"  kpi_transactions_daily: {n_rows:,} randuri agregate")

Construire kpi_transactions_daily...
  kpi_transactions_daily: 43,954 randuri agregate


## 3. Verificare KPI — top 10 zile cu volum maxim

In [0]:
%sql
SELECT 
    date_id,
    SUM(tx_count) AS total_tx,
    ROUND(SUM(tx_volume_total_ron), 0) AS total_volume_ron,
    ROUND(AVG(failure_rate), 1) AS avg_failure_rate
FROM banking.gold.kpi_transactions_daily
GROUP BY date_id
ORDER BY total_volume_ron DESC
LIMIT 10;

date_id,total_tx,total_volume_ron,avg_failure_rate
20251222,137,952622.0,0.8
20260210,136,434209.0,5.2
20250921,115,426363.0,5.4
20250803,124,420856.0,4.5
20250910,152,419593.0,6.0
20250821,129,402481.0,3.8
20250606,141,382769.0,3.8
20250618,129,366115.0,3.2
20250511,140,360055.0,3.7
20250717,143,359703.0,4.1


## 4. `customer_segments_rfm` — Segmentare RFM

**Model RFM:**
- **R**ecency = câte zile au trecut de la ultima tranzacție
- **F**requency = numărul total de tranzacții în ultimele 90 zile
- **M**onetary = volumul total tranzacționat în ultimele 90 zile

**Procedură:**
1. Calculăm cele 3 metrici pentru fiecare client
2. Atribuim cuartile (1-4) — 4 = mai bun, 1 = mai slab
3. Scor RFM = R + F + M (între 3 și 12)
4. Mapăm pe segmente: VIP (10-12), Activ (7-9), Inactiv (4-6), Risc (3)

**Notă:** la calculul cuartilelor, R este invers — recency mică = scor mare (clienți activi recent).

In [0]:
print("Construire customer_segments_rfm...")

# Filtram tranzactiile recente (ultimele 90 zile, doar completed)
fact = spark.table(f"{CATALOG}.{SILVER}.fact_transactions").filter(
    (datediff(current_date(), col("initiated_at")) <= 90) &
    (col("status_code") == "COMPLETED")
)

# Calcul metrici per client
rfm_metrics = (fact.groupBy("customer_id")
    .agg(
        datediff(current_date(), f_max("initiated_at")).alias("recency_days"),
        count("transaction_id").alias("frequency_90d"),
        f_round(f_sum("amount_ron"), 2).alias("monetary_total_ron"),
    )
)

# Adaugam clientii care n-au tranzactii in ultimele 90 zile (dar exista in dim_customers)
all_customers = spark.table(f"{CATALOG}.{GOLD}.dim_customer_enriched").select("customer_id")
rfm_metrics = (all_customers
    .join(rfm_metrics, "customer_id", "left")
    .withColumn("recency_days",       coalesce(col("recency_days"),       lit(999)))
    .withColumn("frequency_90d",      coalesce(col("frequency_90d"),      lit(0)))
    .withColumn("monetary_total_ron", coalesce(col("monetary_total_ron"), lit(0.0)))
)

# Cuartile cu ntile() — atentie: pentru recency, NTILE iese invers
# (recency mica = scor mare = mai bun)
window_r = Window.orderBy(col("recency_days").asc())     # asc pentru ca mic = bun
window_f = Window.orderBy(col("frequency_90d").asc())    # asc pentru ca... vom inversa
window_m = Window.orderBy(col("monetary_total_ron").asc())

rfm_metrics = (rfm_metrics
    .withColumn("r_score_raw", ntile(4).over(window_r))   # 1=mai recent
    .withColumn("f_score_raw", ntile(4).over(window_f))   # 1=mai putine tranzactii
    .withColumn("m_score_raw", ntile(4).over(window_m))   # 1=volum mic
)

# Inversam scorurile pentru a avea convenția "scor mare = mai bun"
rfm_metrics = (rfm_metrics
    .withColumn("r_score", lit(5) - col("r_score_raw"))   # 4=cei mai recenti, 1=cei mai vechi
    .withColumn("f_score", col("f_score_raw"))             # 4=cei mai frecventi
    .withColumn("m_score", col("m_score_raw"))             # 4=volum mai mare
    .drop("r_score_raw", "f_score_raw", "m_score_raw")
)

# Scor RFM total + segment
rfm_metrics = (rfm_metrics
    .withColumn("rfm_score", col("r_score") + col("f_score") + col("m_score"))
    .withColumn("rfm_segment",
        when(col("rfm_score") >= 10, lit("VIP"))
        .when(col("rfm_score") >= 7,  lit("Activ"))
        .when(col("rfm_score") >= 4,  lit("Inactiv"))
        .otherwise(lit("Risc")))
    .withColumn("gold_loaded_at", current_timestamp())
)

# Scriem in Gold
(rfm_metrics.select(
    "customer_id",
    "recency_days", "frequency_90d", "monetary_total_ron",
    "r_score", "f_score", "m_score", "rfm_score",
    "rfm_segment",
    "gold_loaded_at",
).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.customer_segments_rfm"))

n_rows = rfm_metrics.count()
print(f"  customer_segments_rfm: {n_rows} clienti")

Construire customer_segments_rfm...


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  customer_segments_rfm: 500 clienti


## 5. Verificare RFM — distribuție pe segmente

In [0]:
%sql
SELECT 
    rfm_segment,
    COUNT(*) AS n_customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(recency_days), 0) AS avg_recency_days,
    ROUND(AVG(frequency_90d), 1) AS avg_frequency,
    ROUND(AVG(monetary_total_ron), 0) AS avg_monetary_ron
FROM banking.gold.customer_segments_rfm
GROUP BY rfm_segment
ORDER BY 
    CASE rfm_segment 
        WHEN 'VIP' THEN 1 
        WHEN 'Activ' THEN 2 
        WHEN 'Inactiv' THEN 3 
        WHEN 'Risc' THEN 4 
    END;

rfm_segment,n_customers,pct,avg_recency_days,avg_frequency,avg_monetary_ron
VIP,169,33.8,2.0,39.4,53916.0
Activ,138,27.6,5.0,21.1,22118.0
Inactiv,93,18.6,10.0,11.7,9602.0
Risc,100,20.0,950.0,0.3,188.0


## 6. `report_pnl_monthly` — Profit & Loss lunar per branch

**Componente P&L bancar simplificate:**
- **Venituri din comisioane** = suma tranzacțiilor de tip FEE
- **Venituri din dobânzi** = suma tranzacțiilor de tip INTEREST
- **Volum credite disbursate** = suma LOAN_DISBURSEMENT
- **Volum credite rambursate** = suma LOAN_REPAYMENT
- **Volum total tranzacționat** = suma tuturor tranzacțiilor COMPLETED
- **Număr tranzacții procesate** = count COMPLETED

**Profit net estimat = venituri_comisioane + venituri_dobanzi**
(simplificare — într-o bancă reală P&L include și costuri operaționale, provizioane risc, salarii etc.)

In [0]:
print("Construire report_pnl_monthly...")

# Aducem branch_id si timestampul tranzactiilor completate
fact = (spark.table(f"{CATALOG}.{SILVER}.fact_transactions")
    .filter(col("status_code") == "COMPLETED")
    .alias("ft")
    .join(
        spark.table(f"{CATALOG}.{SILVER}.dim_accounts").select("account_id", "branch_id").alias("acc"),
        col("ft.account_id") == col("acc.account_id"),
        "left"
    )
)

# Calculăm year, month din initiated_at
fact = fact.select(
    year(col("ft.initiated_at")).alias("year"),
    month(col("ft.initiated_at")).alias("month"),
    col("acc.branch_id"),
    col("ft.type_code"),
    col("ft.amount_ron"),
    col("ft.transaction_id"),
)

# Agregare per (year, month, branch)
pnl = (fact.groupBy("year", "month", "branch_id")
    .agg(
        # Venituri
        f_sum(when(col("type_code") == "FEE",      col("amount_ron")).otherwise(0)).alias("revenue_fees"),
        f_sum(when(col("type_code") == "INTEREST", col("amount_ron")).otherwise(0)).alias("revenue_interest"),
        
        # Volum credite
        f_sum(when(col("type_code") == "LOAN_DISBURSEMENT", col("amount_ron")).otherwise(0)).alias("loan_disbursed"),
        f_sum(when(col("type_code") == "LOAN_REPAYMENT",    col("amount_ron")).otherwise(0)).alias("loan_repaid"),
        
        # Volum total si count
        f_sum("amount_ron").alias("total_volume_ron"),
        count("transaction_id").alias("total_tx_count"),
    )
)

# Calcule derivate
pnl = (pnl
    .withColumn("total_revenue", col("revenue_fees") + col("revenue_interest"))
    .withColumn("net_profit_estimated", col("total_revenue"))   # simplificat — fara costuri
    
    # Round la valori monetare
    .withColumn("revenue_fees",          f_round(col("revenue_fees"), 2))
    .withColumn("revenue_interest",      f_round(col("revenue_interest"), 2))
    .withColumn("loan_disbursed",        f_round(col("loan_disbursed"), 2))
    .withColumn("loan_repaid",           f_round(col("loan_repaid"), 2))
    .withColumn("total_volume_ron",      f_round(col("total_volume_ron"), 2))
    .withColumn("total_revenue",         f_round(col("total_revenue"), 2))
    .withColumn("net_profit_estimated",  f_round(col("net_profit_estimated"), 2))
    
    # Cheie compusa pentru ușurință de query
    .withColumn("year_month", expr("year * 100 + month"))
    .withColumn("gold_loaded_at", current_timestamp())
)

# Reordonare coloane
pnl = pnl.select(
    "year_month", "year", "month", "branch_id",
    "revenue_fees", "revenue_interest", "total_revenue",
    "loan_disbursed", "loan_repaid",
    "total_volume_ron", "total_tx_count",
    "net_profit_estimated",
    "gold_loaded_at",
)

# Scriem in Gold
(pnl.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.report_pnl_monthly"))

n_rows = pnl.count()
print(f"  report_pnl_monthly: {n_rows} randuri (luna × branch)")

Construire report_pnl_monthly...
  report_pnl_monthly: 260 randuri (luna × branch)


## 7. Verificare P&L — top 5 luni × branch după profit

In [0]:
%sql
SELECT 
    year_month,
    branch_id,
    ROUND(revenue_fees, 0)         AS comisioane,
    ROUND(revenue_interest, 0)     AS dobanzi,
    ROUND(total_revenue, 0)        AS venit_total,
    ROUND(loan_disbursed, 0)       AS credite_acordate,
    ROUND(loan_repaid, 0)          AS credite_rambursate,
    total_tx_count                 AS nr_tranzactii
FROM banking.gold.report_pnl_monthly
ORDER BY total_revenue DESC
LIMIT 10;

year_month,branch_id,comisioane,dobanzi,venit_total,credite_acordate,credite_rambursate,nr_tranzactii
202601,BR-009,49525.0,0.0,49525.0,0.0,14979.0,202
202508,BR-007,20160.0,0.0,20160.0,0.0,3194.0,144
202512,BR-016,19076.0,0.0,19076.0,0.0,26722.0,167
202601,BR-013,18927.0,0.0,18927.0,0.0,14522.0,208
202508,BR-004,15863.0,0.0,15863.0,0.0,12512.0,174
202601,BR-003,14068.0,0.0,14068.0,0.0,18364.0,160
202602,BR-016,13724.0,0.0,13724.0,0.0,16709.0,137
202603,BR-009,13317.0,0.0,13317.0,0.0,14427.0,191
202505,BR-005,13253.0,0.0,13253.0,0.0,25087.0,249
202604,BR-007,12993.0,0.0,12993.0,0.0,2282.0,150


## 8. Verificare P&L — sumar lunar global

In [0]:
%sql
SELECT 
    year_month,
    COUNT(DISTINCT branch_id)         AS branches,
    ROUND(SUM(total_revenue), 0)      AS total_revenue_ron,
    ROUND(SUM(loan_disbursed), 0)     AS total_loans_disbursed,
    SUM(total_tx_count)               AS total_tx
FROM banking.gold.report_pnl_monthly
GROUP BY year_month
ORDER BY year_month;

year_month,branches,total_revenue_ron,total_loans_disbursed,total_tx
202505,20,57258.0,0.0,3723
202506,20,61636.0,0.0,3590
202507,20,51371.0,0.0,3680
202508,20,94223.0,0.0,3634
202509,20,54606.0,0.0,3621
202510,20,68106.0,0.0,3584
202511,20,64070.0,0.0,3466
202512,20,90257.0,0.0,3716
202601,20,146296.0,0.0,3688
202602,20,55521.0,0.0,3280


## 9. Cross-check — agregat KPI pe canal

In [0]:
%sql
SELECT 
    channel_code,
    SUM(tx_count)                  AS total_tx,
    SUM(tx_count_failed)           AS total_failed,
    ROUND(AVG(failure_rate), 2)    AS avg_failure_rate,
    ROUND(SUM(tx_volume_total_ron), 0) AS total_volume
FROM banking.gold.kpi_transactions_daily
GROUP BY channel_code
ORDER BY total_volume DESC;

channel_code,total_tx,total_failed,avg_failure_rate,total_volume
POS,18784,806,4.3,2.1861129E7
ONLINE,12387,533,4.37,1.390106E7
ATM,8875,382,4.26,9693172.0
MOBILE,5924,227,3.83,6975222.0
BRANCH,2559,117,4.33,2685562.0
DIRECT_DEBIT,960,37,3.93,1406746.0


## 10. Validări finale Gold

In [0]:
%sql
-- Sumar global Gold
SELECT 
    'dim_product'              AS tabel, COUNT(*) AS randuri FROM banking.gold.dim_product
UNION ALL SELECT 'dim_customer_enriched',  COUNT(*) FROM banking.gold.dim_customer_enriched
UNION ALL SELECT 'dim_risk_profile',        COUNT(*) FROM banking.gold.dim_risk_profile
UNION ALL SELECT 'kpi_transactions_daily',  COUNT(*) FROM banking.gold.kpi_transactions_daily
UNION ALL SELECT 'customer_segments_rfm',   COUNT(*) FROM banking.gold.customer_segments_rfm
UNION ALL SELECT 'report_pnl_monthly',      COUNT(*) FROM banking.gold.report_pnl_monthly
ORDER BY tabel;

tabel,randuri
customer_segments_rfm,500
dim_customer_enriched,500
dim_product,16
dim_risk_profile,500
kpi_transactions_daily,43954
report_pnl_monthly,260


## Sumar 03b — Gold Aggregates

Realizat:
- **`kpi_transactions_daily`** — granularitate fină (~20-50k rânduri), gata pentru drill-down în dashboards
- **`customer_segments_rfm`** — segmentare RFM cu cuartile + 4 segmente VIP/Activ/Inactiv/Risc
- **`report_pnl_monthly`** — P&L simplificat per lună × branch

**Următorul pas:** `03c_gold_fraud_features` — features pentru modelul ML din MLflow.